# SQL Surgeon Tester Notebook (No Clone Required)

This notebook lets anyone test SQL Surgeon directly from Colab, without cloning the repository.

## What this notebook does

- Connects to the hosted SQL Surgeon environment API.
- Runs one optimization episode (`reset` + repeated `step` calls).
- Uses either:
  - a hosted model provider (`hf` / `openai` / OpenAI-compatible custom), or
  - your fully homemade model logic via a custom stub function.

## Hosted environment URL

- `https://aryadeep232006-sqlsurgeon.hf.space`

## API flow (important)

1. `POST /reset` starts an episode and returns initial observation.
2. `POST /step` sends one action and gets next observation/reward.
3. `GET /state` can be called anytime to inspect episode state.

## Action format expected by `/step`

The server expects the request body shape:

```json
{
  "action": {
    "action_type": "think",
    "query": "",
    "thoughts": "...",
    "confidence": 0.7
  }
}
```

Valid `action_type` values: `think`, `schema`, `explain`, `submit`.

## How to run this notebook

- Run cells top-to-bottom.
- In config, choose `PROVIDER`:
  - `hf` -> uses `HF_TOKEN`
  - `openai` -> uses `OPENAI_API_KEY`
  - `custom` -> uses `MODEL_API_BASE`, `MODEL_NAME`, `MODEL_API_KEY`
- To use a homemade model/agent, set `USE_CUSTOM_DECIDER = True` and implement `decide_action_custom_stub(...)`.

## Common errors

- `401/403` from model provider: wrong or missing provider key.
- `422` from `/step`: payload shape mismatch (must be `{ "action": { ... } }`).
- Non-JSON model output: the notebook has a fallback, but better enforce strict JSON in your prompt/system message.

In [ ]:
!pip -q install requests openai

## 1) Configure your model provider

This section sets both:

- **Environment target** (`BASE_URL`, `TASK_ID`, `MAX_STEPS`)
- **LLM provider settings** (`PROVIDER`, base URL, model name, API key)

### Provider options

- `hf`: Hugging Face Router (OpenAI-compatible)
- `openai`: OpenAI API
- `custom`: any OpenAI-compatible endpoint (self-hosted or third-party)

The notebook first tries environment variables. If no key is found, it asks interactively.

Tip: keep `MAX_STEPS` small while testing integration; increase once stable.

In [ ]:
import os
import json
import requests
from openai import OpenAI

# Persist cookies between /reset and /step (required if the Space uses session cookies).
_http = requests.Session()

# --- Environment API (hosted SQL Surgeon) ---
BASE_URL = "https://aryadeep232006-sqlsurgeon.hf.space"
TASK_ID = "filter_scan"
MAX_STEPS = 5

# --- Provider/model config ---
# Choose one: "hf", "openai", "custom"
PROVIDER = "hf"

if PROVIDER == "hf":
    MODEL_API_BASE = "https://router.huggingface.co/v1"
    MODEL_NAME = "Qwen/Qwen2.5-72B-Instruct"
    MODEL_API_KEY = os.getenv("HF_TOKEN", "")
elif PROVIDER == "openai":
    MODEL_API_BASE = "https://api.openai.com/v1"
    MODEL_NAME = "gpt-4.1-mini"
    MODEL_API_KEY = os.getenv("OPENAI_API_KEY", "")
elif PROVIDER == "custom":
    # Any OpenAI-compatible endpoint
    MODEL_API_BASE = os.getenv("MODEL_API_BASE", "https://your-openai-compatible-endpoint/v1")
    MODEL_NAME = os.getenv("MODEL_NAME", "your-model-name")
    MODEL_API_KEY = os.getenv("MODEL_API_KEY", "")
else:
    raise ValueError("PROVIDER must be one of: hf, openai, custom")

if not MODEL_API_KEY:
    prompt = f"Paste API key for provider '{PROVIDER}' (or set env var first): "
    MODEL_API_KEY = input(prompt).strip()

if not MODEL_API_KEY:
    raise ValueError(
        "Missing API key. Set HF_TOKEN (hf), OPENAI_API_KEY (openai), or MODEL_API_KEY (custom)."
    )

client = OpenAI(base_url=MODEL_API_BASE, api_key=MODEL_API_KEY)
print("Provider:", PROVIDER)
print("Configured model:", MODEL_NAME)
print("Model API base:", MODEL_API_BASE)
print("Env base URL:", BASE_URL)

## 2) Environment client + action policy

This code cell defines:

- `env_reset(...)` and `env_step(...)` wrappers for SQL Surgeon API calls.
- `build_prompt(...)` to convert observation -> model input.
- Action decision pipeline.

### Two decision modes

- `USE_CUSTOM_DECIDER = False`: use provider LLM API directly.
- `USE_CUSTOM_DECIDER = True`: use `decide_action_custom_stub(...)`.

If you built your own model runtime, only edit `decide_action_custom_stub(...)` and return a valid action dict.

### Required returned action fields

- `action_type`: `think`, `schema`, `explain`, or `submit`
- `query`: SQL string (usually required for `submit` and `explain`)
- `thoughts`: short rationale text
- `confidence`: float in `[0, 1]`

In [ ]:
def env_reset(task_id: str):
    resp = _http.post(f"{BASE_URL}/reset", json={"task_id": task_id}, timeout=60)
    resp.raise_for_status()
    return resp.json()


def env_step(action_payload: dict):
    # Required shape: {"action": {...}}
    resp = _http.post(f"{BASE_URL}/step", json={"action": action_payload}, timeout=60)
    resp.raise_for_status()
    return resp.json()


def build_prompt(observation: dict):
    return f"""
You are optimizing SQL for correctness first, then speed.

Task: {observation.get('task_id', '')}
Description: {observation.get('task_description', '')}
Original query:\n{observation.get('original_query', '')}
Schema DDL:\n{observation.get('schema_ddl', '')}

Return ONLY valid JSON with keys:
- action_type: one of [\"think\", \"explain\", \"schema\", \"submit\"]
- query: SQL string (required for submit/explain)
- thoughts: brief rationale
- confidence: number between 0 and 1

Do not wrap JSON in markdown fences. Do not include any text outside the JSON object.
""".strip()


USE_CUSTOM_DECIDER = False


def decide_action_custom_stub(observation: dict):
    """Plug your own model/SDK here.

    TODOs:
    1) Replace this body with your custom model inference call.
    2) Use fields from `observation` to produce the next action.
    3) Return the dict in the exact format below.
    """
    return {
        "action_type": "think",
        "query": "",
        "thoughts": "TODO: replace with your custom model output.",
        "confidence": 0.1,
    }


def decide_action_llm(observation: dict):
    prompt = build_prompt(observation)
    completion = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.2,
        messages=[
            {"role": "system", "content": "Return strict JSON only."},
            {"role": "user", "content": prompt},
        ],
    )
    text = completion.choices[0].message.content.strip()

    # Try direct JSON first
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        # Small fallback: submit original query if model response is malformed.
        parsed = {
            "action_type": "submit",
            "query": observation.get("original_query", ""),
            "thoughts": "Fallback submit due to non-JSON model output.",
            "confidence": 0.2,
        }

    return {
        "action_type": str(parsed.get("action_type", "think")).strip().lower(),
        "query": parsed.get("query", "") or "",
        "thoughts": parsed.get("thoughts", ""),
        "confidence": float(parsed.get("confidence", 0.5)),
    }


def decide_action(observation: dict):
    if USE_CUSTOM_DECIDER:
        return decide_action_custom_stub(observation)
    return decide_action_llm(observation)

## 3) Run one full episode

This cell:

- starts an episode via `/reset`
- optionally runs a short **bootstrap** (`think` → `schema` → `explain`) so you see several steps before any submission
- then calls your model (`decide_action`) repeatedly

### Rewards (important)

On the server, **`think` / `schema` / `explain` steps always have `reward = 0.0`**.  
Only a **`submit`** step is graded, and that is when you can see a **non-zero** `reward` (if the model returns valid JSON and a good optimized query).

To avoid the episode ending immediately on an accidental `submit`, this runner **defers `submit` until `step_idx >= MIN_STEPS_BEFORE_SUBMIT`**.

Tune at the top of the code cell:

- `EXPLORATION_BOOTSTRAP`
- `MIN_STEPS_BEFORE_SUBMIT`
- `MAX_POLICY_STEPS`

You can rerun this cell after changing task/model/settings without reinstalling dependencies.

In [ ]:
# --- Episode runner: longer episodes + graded reward on submit ---
# Tool steps (think/schema/explain) always return reward 0 from the server.
# Only `submit` is graded (non-zero reward possible). We bootstrap tools, then
# defer accidental early `submit` until MIN_STEPS_BEFORE_SUBMIT.

EXPLORATION_BOOTSTRAP = True
MIN_STEPS_BEFORE_SUBMIT = 6  # allow graded submit starting at this step index (1-based after reset)
MAX_POLICY_STEPS = 12  # max LLM-driven steps after bootstrap (episode may end earlier if submit ends it)


def _meta_cumulative_reward(observation: dict):
    meta = observation.get("metadata")
    if isinstance(meta, dict) and "cumulative_reward" in meta:
        return float(meta.get("cumulative_reward") or 0.0)
    return None


result = env_reset(TASK_ID)
obs = result["observation"]
step_result = result
step_idx = 0
running_reward = 0.0

print("Episode started")
print("Task:", obs.get("task_id"))
print("Description:", (obs.get("task_description", "") or "")[:200], "...")

if EXPLORATION_BOOTSTRAP:
    bootstrap_actions = [
        {
            "action_type": "think",
            "query": "",
            "thoughts": "Bootstrap: read task, hints, and constraints before changing SQL.",
            "confidence": 0.5,
        },
        {
            "action_type": "schema",
            "query": "",
            "thoughts": "Bootstrap: pull full schema for join/filter decisions.",
            "confidence": 0.5,
        },
        {
            "action_type": "explain",
            "query": obs.get("original_query", ""),
            "thoughts": "Bootstrap: inspect baseline query plan for the slow query.",
            "confidence": 0.5,
        },
    ]
    for ba in bootstrap_actions:
        step_idx += 1
        step_result = env_step(ba)
        obs = step_result["observation"]
        running_reward += float(step_result.get("reward") or 0.0)
        extra = _meta_cumulative_reward(obs)
        cum_display = f"{extra:.3f}" if extra is not None else f"{running_reward:.3f} (sum of step rewards)"
        print(
            f"step={step_idx} action={ba['action_type']} reward={step_result['reward']:.3f} "
            f"cumulative={cum_display} done={step_result['done']}"
        )
        if step_result["done"]:
            break

for _ in range(MAX_POLICY_STEPS):
    if step_result.get("done"):
        break
    step_idx += 1
    action = decide_action(obs)
    if action.get("action_type") == "submit" and step_idx < MIN_STEPS_BEFORE_SUBMIT:
        action = {
            "action_type": "think",
            "query": "",
            "thoughts": (
                f"Submit deferred (step {step_idx} < {MIN_STEPS_BEFORE_SUBMIT}). "
                "Use schema + explain output, then return strict JSON with action_type=submit "
                "and an optimized query that preserves results."
            ),
            "confidence": 0.3,
        }
    step_result = env_step(action)
    obs = step_result["observation"]
    running_reward += float(step_result.get("reward") or 0.0)
    extra = _meta_cumulative_reward(obs)
    cum_display = f"{extra:.3f}" if extra is not None else f"{running_reward:.3f} (sum of step rewards)"
    print(
        f"step={step_idx} action={action['action_type']} reward={step_result['reward']:.3f} "
        f"cumulative={cum_display} done={step_result['done']}"
    )
    if step_result["done"]:
        break

print("Final step reward:", step_result["reward"])
print("Total reward (sum of step rewards):", f"{running_reward:.3f}")
print("Final observation keys:", list(obs.keys())[:15])